[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndresOrdonez369/deepLearningWeek1/blob/main/proyectos/NB02_sabado_tu_primera_red.ipynb)

# NB02 · Tu primera red: de la cuenta a mano a Keras
**Neural Networks: Deep Learning · UIDE · Clase 1 · Sábado** — Docente: Andrés Ordóñez-Bolaños

Hoy vas a ver **cómo aprende una red completa**. Primero hacemos las cuentas de una red pequeñita (las mismas de las láminas), luego dejamos que TensorFlow las haga por nosotros y terminamos construyendo redes con **Keras** en pocas líneas.

### Cómo usar este notebook
0. Si lo abriste desde GitHub: **Archivo → Guardar una copia en Drive**.
1. Ejecuta la celda **Configuración** (tarda unos segundos en cargar TensorFlow) y sigue en orden con `Shift + Enter`.
2. En cada parte: primero el **Ejemplo resuelto** (ya hecho, compara con **Qué deberías ver**) y después tu **Tarea** (reemplaza los `None` y ejecuta **Revisar**: te dice ✅ o ❌ con una pista).
3. El **Mini-reto 2** está en la Parte F: entrena tu propia red.
4. Las tareas son **formativas** (no tienen nota).

**La red de hoy:** 2 entradas → 2 neuronas ocultas → 1 salida, con sigmoide.

| Parte | Ejemplo resuelto | Tu tarea | Lámina |
|---|---|---|---|
| A | Forward: la red predice | TA · Otro socio, hacia adelante | 51 |
| B | Backward: repartir la culpa | TB · La culpa de la salida | 52–54 |
| C | Un paso de aprendizaje | TC · Un paso con el socio 2 | 55 |
| D | TensorFlow calcula lo mismo | TD · Los engranajes en TensorFlow | 53, 57 |
| E | Una capa oculta resuelve XOR | TE · Cuenta parámetros | 47–49 |
| F | Keras en 3 pasos | 🏁 **Mini-reto 2** · Los círculos | 65–66 |
| G | ¿Qué optimizador llega primero? | TG · Lee las curvas | 62 |

In [ ]:
# @title Configuración y autocorrección · ejecuta esta celda primero (no necesitas abrirla)
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow info messages
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split

SEED = 2026
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------- self-check helpers (formative feedback, no grades) ----------
def _ok(msg):   print("✅ " + msg)
def _no(msg):   print("❌ " + msg)
def _todo():    print("⏳ Aún no completas esta tarea: reemplaza los None y vuelve a ejecutar.")
def _f(v):
    try:
        return np.asarray(v, dtype=float)
    except Exception:
        return None
def _near(a, b, tol=1e-3):
    try:
        return bool(np.allclose(np.asarray(a, dtype=float).reshape(-1), np.asarray(b, dtype=float).reshape(-1), atol=tol))
    except Exception:
        return False
_s = lambda z: 1 / (1 + np.exp(-z))
_W1 = np.array([[0.5, -0.3], [0.2, 0.8]]); _b1 = np.array([0.1, -0.1])
_W2 = np.array([[0.7, -0.4]]);             _b2 = np.array([0.05])
_x2 = np.array([0.2, 1.0]); _y2 = 0.0
_a1 = _s(_W1 @ _x2 + _b1); _yh = _s(_W2 @ _a1 + _b2)
_d2 = _yh - _y2; _dW2 = np.outer(_d2, _a1); _d1 = (_W2.T @ _d2) * _a1 * (1 - _a1); _dW1 = np.outer(_d1, _x2)
_yh_new = _s((_W2 - .5 * _dW2) @ _s((_W1 - .5 * _dW1) @ _x2 + (_b1 - .5 * _d1)) + (_b2 - .5 * _d2))

def revisar_ta(a1_2, y_hat_2, loss_2):
    if a1_2 is None or y_hat_2 is None or loss_2 is None:
        return _todo()
    if not _near(a1_2, _a1):
        return _no(f"La capa oculta debería dar a ≈ {np.round(_a1, 4)}. Recuerda: a = sigmoid(W1 @ x + b1).")
    if not _near(y_hat_2, _yh):
        return _no(f"ŷ debería ser ≈ {_yh[0]:.4f}. Usa a1_2 como entrada de la salida: sigmoid(W2 @ a1_2 + b2).")
    if _near(loss_2, -np.log(_yh)):
        return _no("Ojo: este socio NO pagó (y = 0). La pérdida es -log(1 - ŷ), no -log(ŷ).")
    if not _near(loss_2, -np.log(1 - _yh)):
        return _no(f"La pérdida debería ser -log(1 - ŷ) ≈ {float(-np.log(1 - _yh[0])):.4f}.")
    _ok(f"¡Bien! La red le da {_yh[0]:.0%} de probabilidad de pago a alguien que no pagó: pérdida {float(-np.log(1 - _yh[0])):.3f}.")

def revisar_tb(d2_2, dW2_2):
    if d2_2 is None or dW2_2 is None:
        return _todo()
    if not _near(d2_2, _d2):
        return _no(f"La culpa de la salida es ŷ - y = {_yh[0]:.4f} - 0 = {_d2[0]:.4f}.")
    if not _near(dW2_2, _dW2):
        return _no(f"El gradiente de W2 es culpa × lo que entró: {_d2[0]:.4f} × a1 ≈ {np.round(_dW2.ravel(), 4)}.")
    _ok("Correcto. La culpa es positiva: la red debe BAJAR su predicción para este socio.")

def revisar_tc(y_hat_2_nuevo):
    if y_hat_2_nuevo is None:
        return _todo()
    if not _near(y_hat_2_nuevo, _yh_new):
        return _no(f"Después del paso, ŷ debería ser ≈ {_yh_new[0]:.4f}. ¿Actualizaste las 4 piezas (W1, b1, W2, b2) con η = 0.5?")
    _ok(f"¡La red aprendió! ŷ bajó de {_yh[0]:.4f} a {_yh_new[0]:.4f}: ahora duda más de que este socio pague, como debe ser.")

def revisar_td(grad_engranajes):
    if grad_engranajes is None:
        return _todo()
    if not _near(grad_engranajes, 18.0):
        return _no(f"La derivada de (3x)² en x = 1 es 18 (×3 por dentro, ×6 por fuera). Obtuviste {float(_f(grad_engranajes))}.")
    _ok("18 = 3 × 6: TensorFlow aplicó la regla de la cadena, como los engranajes.")

def revisar_te(contar_parametros):
    if contar_parametros is None:
        return _todo()
    try:
        r = [contar_parametros(c) for c in ([2, 16, 16, 1], [28, 32, 16, 1], [3, 4, 1])]
    except Exception as e:
        return _no(f"Tu función produjo un error: {e}")
    if r[0] is None:
        return _todo()
    if r != [337, 1473, 21]:
        return _no(f"Esperaba 337, 1473 y 21; obtuve {r}. Pista: en cada capa, entradas × neuronas + neuronas.")
    _ok("337 (red de las lunas), 1 473 (red del mini-proyecto) y 21. Igual que model.summary().")

def revisar_mini_reto_2(modelo, X_va, y_va):
    if modelo is None:
        return _todo()
    try:
        acc = float(np.mean((modelo.predict(X_va, verbose=0).ravel() >= 0.5) == y_va))
    except Exception as e:
        return _no(f"No pude evaluar tu modelo: {e}")
    print(f"Exactitud en validación: {acc:.1%}")
    if acc >= 0.90:
        _ok("¡Superaste el 90 %! Tu red aprendió una frontera circular.")
    elif acc < 0.6:
        _no("Cerca del 50 % = adivinar. ¿Tu red tiene capas ocultas con activación (relu o tanh)?")
    else:
        _no("Vas bien. Prueba más neuronas, más épocas o learning_rate=0.01.")

def revisar_tg(peor_optimizador, resultados):
    if peor_optimizador is None:
        return _todo()
    worst = max(resultados, key=resultados.get)
    if str(peor_optimizador).strip().lower() != worst.lower():
        return _no(f"Mira el final de las curvas (o el diccionario resultados_g): ¿cuál quedó más arriba?")
    _ok(f"Correcto: {worst} terminó con la pérdida más alta. Sin impulso ni pasos adaptativos, avanza lento.")

def plot_boundary(model, X, y, ax, title):
    g1, g2 = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    P = model.predict(np.c_[g1.ravel(), g2.ravel()], verbose=0).reshape(g1.shape)
    ax.contourf(g1, g2, P, levels=20, cmap="RdBu_r", alpha=.55)
    ax.contour(g1, g2, P, levels=[.5], colors="k", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu_r", edgecolor="k", s=30); ax.set_title(title)

print("Listo ✔  TensorFlow", tf.__version__, "| Keras", keras.__version__, "· autocorrección cargada")

---
## Parte A · Forward: la información viaja hacia adelante
**Idea:** cada capa hace lo mismo que una neurona del viernes, pero con varias neuronas a la vez:

$$\mathbf{a}^{(1)} = \sigma\big(W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}\big) \qquad \hat y = \sigma\big(W^{(2)}\mathbf{a}^{(1)} + b^{(2)}\big)$$

Cada **fila** de $W$ son los pesos de una neurona.

### Ejemplo resuelto: el socio 1
$x_1$ = antigüedad (1.0), $x_2$ = carga de la cuota (0.5), ambas escaladas. **La verdad:** $y = 1$ (sí pagó).

| Capa | Pesos | Sesgos |
|---|---|---|
| Oculta | neurona 1: (0.5, −0.3) · neurona 2: (0.2, 0.8) | 0.1 · −0.1 |
| Salida | (0.7, −0.4) | 0.05 |

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))

x  = np.array([1.0, 0.5]); y = 1.0
W1 = np.array([[0.5, -0.3],     # hidden neuron 1
               [0.2,  0.8]])    # hidden neuron 2
b1 = np.array([0.1, -0.1])
W2 = np.array([[0.7, -0.4]])    # output neuron
b2 = np.array([0.05])

z1 = W1 @ x + b1; a1 = sigmoid(z1)      # hidden layer
z2 = W2 @ a1 + b2; y_hat = sigmoid(z2)  # output layer
loss = -np.log(y_hat)                    # cross-entropy for y = 1

print("Capa oculta:  z =", z1, " -> a =", a1)
print("Salida:       z =", z2, " -> ŷ =", y_hat)
print("Pérdida:      -log(ŷ) =", loss)

**Qué deberías ver:** capa oculta a = (0.6106, 0.6225); predicción **ŷ = 0.5569** (56 % de probabilidad de pagar); pérdida **0.5854**.

### Tarea A · Otro socio, hacia adelante
El **socio 2** tiene antigüedad 0.2 y carga de cuota 1.0, y **no pagó** ($y = 0$). Con la **misma red**:
1. Calcula la capa oculta → `a1_2`.
2. Calcula la predicción → `y_hat_2`.
3. Calcula la pérdida. Ojo: como $y = 0$, la BCE es $-\log(1-\hat y)$ → `loss_2`.

In [ ]:
x_2 = np.array([0.2, 1.0]); y_2 = 0.0
a1_2 = None       # TODO: sigmoid(W1 @ x_2 + b1)
y_hat_2 = None    # TODO: sigmoid(W2 @ a1_2 + b2)
loss_2 = None     # TODO: -log(1 - y_hat_2)

In [ ]:
revisar_ta(a1_2, y_hat_2, loss_2)

---
## Parte B · Backward: la culpa viaja hacia atrás
**Idea:** el error se descubre al final y se reparte hacia atrás, como en la cadena del cacao: el cliente dice "salió amargo" y cada eslabón recibe culpa según cuánto influyó. Tres reglas bastan:

1. **Culpa de la salida:** $\delta^{(2)} = \hat y - y$.
2. **Culpa de una neurona oculta:** la culpa que le llega por su peso × la pendiente de su activación, $\sigma'(z) = a(1-a)$.
3. **Gradiente de un peso:** culpa de su neurona × lo que entró por ese cable.

### Ejemplo resuelto: el socio 1

In [ ]:
d2  = y_hat - y                       # rule 1: output error
dW2 = np.outer(d2, a1); db2 = d2      # rule 3 for the output layer

d1  = (W2.T @ d2) * a1 * (1 - a1)     # rule 2: error sent back, times sigmoid slope
dW1 = np.outer(d1, x);  db1 = d1      # rule 3 for the hidden layer

print("Culpa de la salida   δ2 =", d2)
print("Gradiente W2         =", dW2)
print("Culpa capa oculta    δ1 =", d1)
print("Gradiente W1         =\n", dW1)

**Qué deberías ver:** δ2 = **−0.4431** (la predicción se quedó corta); gradiente de W2 = (−0.2706, −0.2758); δ1 = (**−0.0738**, **0.0417**).
La culpa de la capa oculta es mucho más pequeña: la pendiente de la sigmoide (≈ 0.24) la encoge al pasar hacia atrás.

### Tarea B · La culpa de la salida (socio 2)
Con los resultados de tu Tarea A:
1. Culpa de la salida → `d2_2` (regla 1).
2. Gradiente de los pesos de la salida → `dW2_2` (regla 3: `np.outer(d2_2, a1_2)`).

In [ ]:
d2_2 = None     # TODO: rule 1
dW2_2 = None    # TODO: rule 3 for the output layer

In [ ]:
revisar_tb(d2_2, dW2_2)

---
## Parte C · Un paso de aprendizaje y su verificación
**Idea:** mover cada peso **en contra** de su gradiente: $\;w \leftarrow w - \eta \cdot \text{gradiente}$. Luego verificamos los gradientes con un método independiente (*gradient check*).

### Ejemplo resuelto: el socio 1, con η = 0.5

In [ ]:
eta = 0.5
W1n, b1n = W1 - eta * dW1, b1 - eta * db1
W2n, b2n = W2 - eta * dW2, b2 - eta * db2
y_hat_new = sigmoid(W2n @ sigmoid(W1n @ x + b1n) + b2n)
print(f"Predicción: {y_hat[0]:.4f} -> {y_hat_new[0]:.4f}")
print(f"Pérdida:    {loss[0]:.4f} -> {-np.log(y_hat_new[0]):.4f}")

# Gradient check: nudge each weight a tiny bit and measure how the loss changes
def loss_of(params):
    W1_, b1_, W2_, b2_ = params[:4].reshape(2, 2), params[4:6], params[6:8].reshape(1, 2), params[8:]
    return float(-np.log(sigmoid(W2_ @ sigmoid(W1_ @ x + b1_) + b2_)[0]))
p0 = np.concatenate([W1.ravel(), b1, W2.ravel(), b2])
g_backprop = np.concatenate([dW1.ravel(), db1, dW2.ravel(), db2])
g_numeric = np.array([(loss_of(p0 + 1e-5 * e) - loss_of(p0 - 1e-5 * e)) / 2e-5 for e in np.eye(len(p0))])
rel_err = np.linalg.norm(g_numeric - g_backprop) / (np.linalg.norm(g_numeric) + np.linalg.norm(g_backprop))
print(f"Gradient check: error relativo = {rel_err:.1e}  (menor que 1e-6 = correcto)")

**Qué deberías ver:** predicción 0.5569 → **0.6542** y pérdida 0.5854 → **0.4243** (bajó 27 %). El gradient check da ~**1e-11**: las cuentas son correctas.

### Tarea C · Un paso con el socio 2
1. Calcula la culpa de la capa oculta del socio 2 (regla 2) y su gradiente `dW1_2` (regla 3).
2. Actualiza **las cuatro piezas** (W1, b1, W2, b2) con η = 0.5.
3. Calcula la nueva predicción para el socio 2 → `y_hat_2_nuevo`. ¿Subió o bajó? ¿Tiene sentido?

<details><summary>Pista</summary>

Copia el ejemplo de las Partes B y C y cambia `x → x_2`, `a1 → a1_2`, `d2 → d2_2`. El gradiente de cada sesgo es su culpa (`db = d`).
</details>

In [ ]:
d1_2 = None       # TODO: rule 2
dW1_2 = None      # TODO: rule 3 for the hidden layer
# TODO: update W1, b1, W2, b2 with eta = 0.5 (use new names, e.g. W1_2n)

y_hat_2_nuevo = None    # TODO: forward pass of x_2 with the updated weights

In [ ]:
revisar_tc(y_hat_2_nuevo)

---
## Parte D · TensorFlow hace la cuenta por ti
**Idea:** `tf.GradientTape` funciona como una grabadora: graba todo lo que pasa hacia adelante y luego lo recorre al revés aplicando la regla de la cadena. Así entrena cualquier red de verdad.

### Ejemplo resuelto: los mismos gradientes del socio 1

In [ ]:
tW1, tb1, tW2, tb2 = [tf.Variable(v) for v in (W1, b1, W2, b2)]
with tf.GradientTape() as tape:                     # the tape "records" every operation
    t_hidden = tf.sigmoid(tf.linalg.matvec(tW1, tf.constant(x)) + tb1)
    t_yhat = tf.sigmoid(tf.linalg.matvec(tW2, t_hidden) + tb2)
    t_loss = -tf.math.log(t_yhat[0])
gW1, gb1, gW2, gb2 = tape.gradient(t_loss, [tW1, tb1, tW2, tb2])

print("Gradiente W1 según TensorFlow:\n", gW1.numpy())
diff = max(np.abs(gW1.numpy() - dW1).max(), np.abs(gW2.numpy() - dW2).max())
print("Diferencia máxima con nuestra cuenta a mano:", diff)

**Qué deberías ver:** la misma matriz de la Parte B y una diferencia del orden de **1e-17** (cero en la práctica).

### Tarea D · Los engranajes en TensorFlow
En las láminas vimos $z = (3x)^2$: por dentro se multiplica por 3 y por fuera, en $x = 1$, por 6. Usa `GradientTape` para obtener $\dfrac{dz}{dx}$ en $x = 1$ → `grad_engranajes`.

<details><summary>Pista</summary>

```python
xv = tf.Variable(1.0)
with tf.GradientTape() as tape:
    zv = (3 * xv) ** 2
grad = tape.gradient(zv, xv)
```
</details>

In [ ]:
grad_engranajes = None    # TODO: use tf.GradientTape

In [ ]:
revisar_td(grad_engranajes)

---
## Parte E · Una capa oculta resuelve XOR (y las lunas)
**Idea:** el perceptrón del viernes no pudo con XOR porque solo traza una línea. Con una **capa oculta**, cada neurona traza su propia línea y la salida las combina.

### Ejemplo resuelto

In [ ]:
# XOR with one hidden layer of 8 neurons
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float); y_xor = np.array([0, 1, 1, 0], float)
keras.utils.set_random_seed(SEED)
xor_net = keras.Sequential([keras.Input((2,)), keras.layers.Dense(8, activation="tanh"),
                            keras.layers.Dense(1, activation="sigmoid")])
xor_net.compile(optimizer=keras.optimizers.Adam(0.05), loss="binary_crossentropy")
xor_net.fit(X_xor, y_xor, epochs=500, batch_size=4, verbose=0)
print("Predicciones XOR:", xor_net.predict(X_xor, verbose=0).ravel().round(2), "(esperado: 0 1 1 0)")

# Moons with two hidden layers
X_m, y_m = make_moons(600, noise=0.2, random_state=SEED)
moons_net = keras.Sequential([keras.Input((2,)), keras.layers.Dense(16, activation="relu"),
                              keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1, activation="sigmoid")])
moons_net.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
moons_net.fit(X_m, y_m, epochs=60, batch_size=32, verbose=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
plot_boundary(xor_net, X_xor, y_xor, ax[0], "XOR resuelto con 1 capa oculta")
plot_boundary(moons_net, X_m, y_m, ax[1], "Lunas: la frontera se curva")
plt.tight_layout(); plt.show()

**Qué deberías ver:** predicciones de XOR **0, 1, 1, 0** y una frontera curva que separa bien las dos lunas.

### Tarea E · Cuenta parámetros
Escribe `contar_parametros(capas)`, que recibe una lista como `[2, 16, 16, 1]` (entradas, ocultas…, salida) y devuelve cuántos números aprende la red.

Regla por capa: **entradas × neuronas + neuronas** (un peso por conexión y un sesgo por neurona).

<details><summary>Pista</summary>

Recorre parejas consecutivas: `for n_in, n_out in zip(capas[:-1], capas[1:]):`
</details>

In [ ]:
def contar_parametros(capas):
    return None    # TODO

In [ ]:
revisar_te(contar_parametros)

---
## Parte F · Keras en 3 pasos: construir, compilar, entrenar
**Idea:** todo lo anterior (forward, backward, actualizar, por lotes y épocas) Keras lo hace en tres pasos:

1. **Construir:** apilar capas (`Sequential`).
2. **Compilar:** elegir optimizador, pérdida y métrica.
3. **Entrenar:** `fit` repite adelante → atrás → actualizar.

### Ejemplo resuelto: la red de las lunas

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X_m, y_m, test_size=0.25, random_state=SEED, stratify=y_m)

# 1) BUILD
model = keras.Sequential([
    keras.Input(shape=(2,)),                                                    # 2 input features
    keras.layers.Dense(16, activation="relu", kernel_initializer="he_normal"),  # hidden layer 1
    keras.layers.Dense(16, activation="relu", kernel_initializer="he_normal"),  # hidden layer 2
    keras.layers.Dense(1, activation="sigmoid"),                                # probability of class 1
])
model.summary()

# 2) COMPILE
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
              loss="binary_crossentropy", metrics=["accuracy"])

# 3) TRAIN
history = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=60, batch_size=32, verbose=0)
print(f"Exactitud en validación: {history.history['val_accuracy'][-1]:.1%}")

plt.figure(figsize=(6.5, 3.6))
plt.plot(history.history["loss"], label="entrenamiento"); plt.plot(history.history["val_loss"], label="validación")
plt.xlabel("época"); plt.ylabel("pérdida"); plt.legend(); plt.title("La pérdida baja época a época"); plt.show()

**Qué deberías ver:** un resumen con **337 parámetros**, exactitud de validación cercana a **95 %** y dos curvas que bajan juntas.

---
## 🏁 Mini-reto 2 · Los círculos
Un problema nuevo: puntos de una clase **dentro** de un círculo y de la otra clase **alrededor**. Ninguna línea recta los separa.

**Tu misión:** construye, compila y entrena una red con Keras que llegue a **≥ 90 % de exactitud en validación**.

**Para pensar:**
1. Primero prueba una red **sin capas ocultas** (solo `Dense(1, activation="sigmoid")`). ¿Qué exactitud logra? ¿Por qué?
2. Luego agrega capas ocultas. ¿Cuántas neuronas necesitas como mínimo?

In [ ]:
X_c, y_c = make_circles(600, noise=0.1, factor=0.5, random_state=SEED)
Xc_tr, Xc_va, yc_tr, yc_va = train_test_split(X_c, y_c, test_size=0.25, random_state=SEED, stratify=y_c)
plt.figure(figsize=(4.5, 4.5)); plt.scatter(*X_c.T, c=y_c, cmap="RdBu_r", edgecolor="k", s=25)
plt.title("Los círculos: ¿cómo los separas?"); plt.show()

In [ ]:
keras.utils.set_random_seed(SEED)
modelo_circulos = None
# TODO 1) BUILD:   modelo_circulos = keras.Sequential([keras.Input(shape=(2,)), ..., keras.layers.Dense(1, activation="sigmoid")])
# TODO 2) COMPILE: modelo_circulos.compile(optimizer=..., loss="binary_crossentropy", metrics=["accuracy"])
# TODO 3) TRAIN:   hist_c = modelo_circulos.fit(Xc_tr, yc_tr, validation_data=(Xc_va, yc_va), epochs=60, batch_size=32, verbose=0)

In [ ]:
revisar_mini_reto_2(modelo_circulos, Xc_va, yc_va)
if modelo_circulos is not None:
    fig, ax = plt.subplots(figsize=(4.8, 4.5)); plot_boundary(modelo_circulos, X_c, y_c, ax, "Tu frontera"); plt.show()

---
## Parte G · ¿Qué optimizador llega primero?
**Idea:** el optimizador es la estrategia para bajar la montaña.
- **SGD:** el caminante. Pasos fijos en la dirección de la pendiente.
- **Momentum:** la bicicleta en bajada. Acumula impulso.
- **Adam:** el guía con experiencia. Ajusta el paso de cada peso por separado.

### Ejemplo resuelto: misma red, mismos pesos iniciales, tres optimizadores

In [ ]:
def make_model():
    keras.utils.set_random_seed(SEED)   # same starting weights for a fair comparison
    return keras.Sequential([keras.Input((2,)), keras.layers.Dense(16, activation="relu"),
                             keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1, activation="sigmoid")])

optimizers = {"SGD": keras.optimizers.SGD(0.05),
              "Momentum": keras.optimizers.SGD(0.05, momentum=0.9),
              "Adam": keras.optimizers.Adam(0.01)}
resultados_g = {}
plt.figure(figsize=(7, 3.8))
for name, opt in optimizers.items():
    m = make_model(); m.compile(optimizer=opt, loss="binary_crossentropy")
    h = m.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=40, batch_size=32, verbose=0)
    resultados_g[name] = round(h.history["val_loss"][-1], 3)
    plt.plot(h.history["val_loss"], lw=2, label=name)
plt.xlabel("época"); plt.ylabel("pérdida de validación"); plt.legend(); plt.title("Misma red, tres optimizadores"); plt.show()

**Qué deberías ver:** SGD baja despacio; Momentum y Adam llegan mucho antes a una pérdida baja.

### Tarea G · Lee las curvas
¿Qué optimizador terminó con la pérdida de validación **más alta**? Escribe `"SGD"`, `"Momentum"` o `"Adam"` en `peor_optimizador`.

In [ ]:
peor_optimizador = None    # TODO

In [ ]:
revisar_tg(peor_optimizador, resultados_g)

---
### Para llevar
1. **Forward** calcula la predicción; **backward** reparte la culpa hacia atrás con la regla de la cadena; el **paso** mueve cada peso en contra de su gradiente.
2. TensorFlow hace esas derivadas solo (`GradientTape`) y Keras lo envuelve todo en `fit`.
3. Las capas ocultas permiten fronteras curvas; el optimizador decide qué tan rápido llegamos.

**Retos opcionales:** (1) en la Parte F, cambia `relu` por `tanh`; (2) en la Parte G, prueba `SGD(0.5)`; (3) entrena las lunas con `noise=0.35`.

**Siguiente:** abre el **NB03** para el mini-proyecto de la cooperativa.